## Who Worked On This Part?
Aarya Ray Chaudhuri: turned the cleaning from `2MushroomClean.ipynb` into this short version.

# Mushroom data preparation (no graphs)
This notebook goes from the raw data to the prepared data in one go: only the cleaning steps we decided to keep, no graphs and no comparisons. **Why** each step is here, and the tests that prove it helps, are in `2MushroomClean.ipynb`. Each step below links back to the matching step there.

Input: `datasets/mushroom_project_dataset.csv` → output: `datasets/mushroom_clean.csv`

## Paths
We look for the repo folder starting from wherever the notebook runs and build the full path from there. That way it works on every laptop in the team and in the deployment pipeline, without a hardcoded `C:\Users\...` path.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# walk up from where the notebook runs to the repo root, so the data path works on every machine
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Mushroom' / 'datasets').is_dir())
DATA = ROOT / 'Mushroom' / 'datasets'
DATA.relative_to(ROOT)

WindowsPath('Mushroom/datasets')

## Load

In [2]:
raw = pd.read_csv(DATA / 'mushroom_project_dataset.csv')
df = raw.copy()
df.shape

(5000, 13)

## Column names and target
Underscores instead of dashes, and the unit in the name of every measurement (stem width is in mm, the rest in cm). `class` becomes `is_poisonous`, with 1 = poisonous because that's the class we must not miss.

In [3]:
df.columns = df.columns.str.replace('-', '_')
df = df.rename(columns={'cap_diameter': 'cap_diameter_cm', 'stem_height': 'stem_height_cm', 'stem_width': 'stem_width_mm'})
df['is_poisonous'] = (df.pop('class') == 'p').astype(int)

num_cols = ['cap_diameter_cm', 'stem_height_cm', 'stem_width_mm']

## Step 1: drop the noise columns
`jumbled_noise_0` and `jumbled_noise_1` are shuffled copies of cap_shape. Removing them made the forest better in all 15 folds (clean notebook, step 1).

In [4]:
df = df.drop(columns=['jumbled_noise_0', 'jumbled_noise_1'])

## Step 2: duplicates are kept
There's no code for this step on purpose. The 5 duplicate rows are mostly-empty rows of different mushrooms, not real copies (clean notebook, step 2).

## Step 3: mushrooms without a stem
stem_surface f (none) always goes with a stem of 0, so they fill each other's gaps (clean notebook, step 3).

In [5]:
# a 0 stem means no stem surface, and no stem surface means both stem measurements are 0
zero_stem = (df['stem_height_cm'] == 0) | (df['stem_width_mm'] == 0)
df.loc[zero_stem, 'stem_surface'] = 'f'
df.loc[df['stem_surface'] == 'f', ['stem_height_cm', 'stem_width_mm']] = 0

## Step 4: "missing" as its own value in the categorical columns
Better than filling in the most common value, and much better than dropping spore_print_color and stem_surface (clean notebook, step 4).

In [6]:
cat_cols = df.columns.drop(num_cols + ['is_poisonous']).tolist()
df[cat_cols] = df[cat_cols].fillna('missing')

## Step 5: estimate the missing measurements from each other
An iterative imputer on the log1p scale predicts each measurement from the other two. For cap diameter that halves the error compared to the median (clean notebook, step 5). `max_iter=25` because the default of 10 rounds doesn't converge here.

In [7]:
imputer = IterativeImputer(max_iter=25, random_state=42)
df[num_cols] = np.expm1(imputer.fit_transform(np.log1p(df[num_cols]))).clip(min=0).round(2)

## Step 6: no capping
No code here either. The big values are a real giant species and capping didn't help (clean notebook, step 6). log1p only helps linear models, so it goes in their pipeline in the predict notebook, not in this file.

## Step 7: data types
All categoricals are unordered (season is a cycle, not a scale), so they get the plain `category` type. The CSV doesn't keep this type, so set it again after loading.

In [8]:
df[cat_cols] = df[cat_cols].astype('category')

## Step 8: labels stay as they are
The likely flipped labels (4 stemless edible, 3 giant poisonous) are not changed (clean notebook, step 8).

## Checks and save
Before saving we make sure the result is what we expect. If any of these fail, something in the steps above changed.

In [9]:
assert df.isna().sum().sum() == 0, 'there are still missing values'
assert len(df) == len(raw), 'rows got lost'
assert (df[num_cols] >= 0).all().all(), 'negative measurements'
assert df['is_poisonous'].sum() == (raw['class'] == 'p').sum(), 'labels changed'

df.to_csv(DATA / 'mushroom_clean.csv', index=False)
print(df.shape)
df.head()

(5000, 11)


,cap_diameter_cm,stem_height_cm,stem_width_mm,spore_print_color,gill_color,habitat,season,ring_type,cap_shape,stem_surface,is_poisonous
0,7.25,6.42,13.51,missing,missing,d,u,g,x,missing,1
1,6.99,7.68,18.76,missing,w,d,a,f,f,s,0
2,5.78,7.67,8.21,missing,r,d,u,f,c,missing,1
3,6.34,8.34,11.06,missing,missing,d,a,z,x,missing,1
4,7.01,5.67,12.78,missing,n,d,missing,f,x,missing,0


## Summary
From 5000 raw rows with 12 features (16,300 empty cells) to 5000 rows with 10 features and no empty cells:
- 2 noise columns removed
- 18 stem values filled with the stemless rule
- the gaps in the categories marked as "missing" instead of guessed
- about 3,400 missing measurements estimated from the other measurements
- nothing capped, no rows dropped, no labels changed

The output is exactly the same as what `2MushroomClean.ipynb` produces, because both use the same steps and the same `random_state`. If a step changes in the clean notebook, it has to change here too.